In [2]:
import pandas as pd
import shutil
import os
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt

# Import required libraries for semantic segmentation
import sys
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation
import torch.nn.functional as F

# 1. Extract Images and Save to a Folder

This section extracts specific images from the Place Pulse 2.0 dataset based on location IDs found in a CSV file and saves them to a designated output folder.

In [3]:
# Set paths
csv_path = r'E:\01_UCL\08_Dissertation\02_DataProcessing\01_SpaceSyntax_London\London_RegressionModel_wgs84.csv'
image_folder = r'E:\01_UCL\08_Dissertation\01_DataCollection\01 Place Pluse2.0 Dataset\00 place-pulse-dataset-images _OriginalDatasdet\images'
output_folder = r'E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\Selected_Images'

# Create output folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

# Read CSV and extract location_id list
df = pd.read_csv(csv_path)
location_ids = set(df['location_id'].tolist())

print(len(location_ids), "unique location_ids found in the CSV.")

# Match and copy images
matched = 0
unmatched_ids = set(location_ids)

for filename in os.listdir(image_folder):
    for loc_id in location_ids:
        if loc_id in filename:
            src = os.path.join(image_folder, filename)
            dst = os.path.join(output_folder, filename)
            shutil.copy(src, dst)
            matched += 1
            unmatched_ids.discard(loc_id)
            break  # Stop checking once matched

print(f" Total matched and copied images: {matched}")
print(f" Total unmatched location_ids: {len(unmatched_ids)}")

900 unique location_ids found in the CSV.
 Total matched and copied images: 900
 Total unmatched location_ids: 0


# 2. Image Segmentation

This section implements semantic segmentation using the SegFormer model to calculate green view ratio and sky visibility from street view images.

In [4]:
# Configure semantic segmentation model - Using SegFormer model
# This model performs excellently in urban landscape segmentation, 
# particularly suitable for calculating green view ratio and sky visibility

# Set up model and processor - Using model that supports safetensors
model_name = "nvidia/segformer-b0-finetuned-ade-512-512"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForSemanticSegmentation.from_pretrained(model_name, use_safetensors=True)

# Main category labels for ADE20K dataset (partial)
# Categories we focus on:
# - tree (4): Trees
# - grass (9): Grass
# - sky (2): Sky
# - plant (17): Plants

# Define category indices we focus on (ADE20K)
TREE_CLASS = 4       # Tree category
SKY_CLASS = 2        # Sky category
GRASS_CLASS = 9      # Grass category
PLANT_CLASS = 17     # Plant category

print("Model loaded successfully!")
print(f"Categories of interest (ADE20K dataset):")
print(f"- Trees (tree): Index {TREE_CLASS}")
print(f"- Sky (sky): Index {SKY_CLASS}")
print(f"- Grass (grass): Index {GRASS_CLASS}")
print(f"- Plants (plant): Index {PLANT_CLASS}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Model loaded successfully!
Categories of interest (ADE20K dataset):
- Trees (tree): Index 4
- Sky (sky): Index 2
- Grass (grass): Index 9
- Plants (plant): Index 17


In [5]:
def calculate_green_view_ratio_and_sky_visibility(image_path):
    """
    Calculate green view ratio and sky visibility for an image
    
    Args:
        image_path (str): Path to the image file
    
    Returns:
        dict: Dictionary containing green view ratio, sky visibility and other metrics
    """
    try:
        # Load image
        image = Image.open(image_path).convert("RGB")
        original_size = image.size
        
        # Preprocess image
        inputs = processor(images=image, return_tensors="pt")
        
        # Perform semantic segmentation prediction
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        
        # Resize prediction results to original image size
        upsampled_logits = F.interpolate(
            logits,
            size=image.size[::-1],  # PIL size is (width, height), need to convert to (height, width)
            mode="bilinear",
            align_corners=False,
        )
        
        # Get predicted categories
        predicted = upsampled_logits.argmax(dim=1).squeeze().cpu().numpy()
        
        # Calculate total pixels
        total_pixels = predicted.shape[0] * predicted.shape[1]
        
        # Calculate pixel counts for each category (ADE20K)
        tree_pixels = np.sum(predicted == TREE_CLASS)
        sky_pixels = np.sum(predicted == SKY_CLASS)
        grass_pixels = np.sum(predicted == GRASS_CLASS)
        plant_pixels = np.sum(predicted == PLANT_CLASS)
        
        # Calculate Green View Ratio
        # Includes trees, grass and plants
        green_pixels = tree_pixels + grass_pixels + plant_pixels
        green_view_ratio = (green_pixels / total_pixels) * 100
        
        # Calculate Sky Visibility
        sky_visibility = (sky_pixels / total_pixels) * 100
        
        # Calculate pure tree ratio
        tree_ratio = (tree_pixels / total_pixels) * 100
        
        return {
            'image_path': image_path,
            'image_name': os.path.basename(image_path),
            'total_pixels': total_pixels,
            'tree_pixels': tree_pixels,
            'sky_pixels': sky_pixels,
            'grass_pixels': grass_pixels,
            'plant_pixels': plant_pixels,
            'green_view_ratio': round(green_view_ratio, 2),
            'sky_visibility': round(sky_visibility, 2),
            'tree_ratio': round(tree_ratio, 2),
            'image_width': original_size[0],
            'image_height': original_size[1]
        }
        
    except Exception as e:
        print(f"Error processing image {image_path}: {str(e)}")
        return None

def visualize_segmentation(image_path, save_visualization=False):
    """
    Visualize semantic segmentation results
    
    Args:
        image_path (str): Path to the image file
        save_visualization (bool): Whether to save visualization results
    """
    try:
        # Load image
        image = Image.open(image_path).convert("RGB")
        
        # Preprocess image
        inputs = processor(images=image, return_tensors="pt")
        
        # Perform semantic segmentation prediction
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        
        # Resize prediction results to original image size
        upsampled_logits = F.interpolate(
            logits,
            size=image.size[::-1],
            mode="bilinear",
            align_corners=False,
        )
        
        # Get predicted categories
        predicted = upsampled_logits.argmax(dim=1).squeeze().cpu().numpy()
        
        # Create color mapping
        # Assign different colors for different categories (ADE20K)
        color_map = np.zeros((150, 3), dtype=np.uint8)  # ADE20K has 150 categories
        color_map[TREE_CLASS] = [0, 128, 0]          # Dark green - Trees
        color_map[SKY_CLASS] = [135, 206, 235]       # Sky blue - Sky
        color_map[GRASS_CLASS] = [124, 252, 0]       # Lawn green - Grass
        color_map[PLANT_CLASS] = [34, 139, 34]       # Forest green - Plants
        
        # Create colored segmentation map
        segmentation_colored = color_map[predicted]
        
        # Display results
        fig, axes = plt.subplots(1, 2, figsize=(15, 7))
        
        # Original image
        axes[0].imshow(image)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Segmentation results
        axes[1].imshow(segmentation_colored)
        axes[1].set_title('Semantic Segmentation Results')
        axes[1].axis('off')
        
        # Add legend
        legend_elements = [
            plt.Rectangle((0, 0), 1, 1, facecolor=np.array([0, 128, 0])/255, label='Trees'),
            plt.Rectangle((0, 0), 1, 1, facecolor=np.array([135, 206, 235])/255, label='Sky'),
            plt.Rectangle((0, 0), 1, 1, facecolor=np.array([124, 252, 0])/255, label='Grass'),
            plt.Rectangle((0, 0), 1, 1, facecolor=np.array([34, 139, 34])/255, label='Plants')
        ]
        axes[1].legend(handles=legend_elements, loc='upper right')
        
        plt.tight_layout()
        
        if save_visualization:
            save_path = image_path.replace('.jpg', '_segmentation.png').replace('.jpeg', '_segmentation.png')
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Visualization results saved to: {save_path}")
        
        plt.show()
        
    except Exception as e:
        print(f"Error visualizing image {image_path}: {str(e)}")

print("Image processing functions defined successfully!")

Image processing functions defined successfully!


In [6]:
# Batch process images and generate CSV file
def process_all_images_and_generate_csv():
    """
    Batch process all images, calculate green view ratio and sky visibility, and generate CSV file
    """
    # Set image folder path
    image_folder = r'E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\Selected_Images'
    
    # Check if folder exists
    if not os.path.exists(image_folder):
        print(f"Error: Image folder does not exist: {image_folder}")
        return
    
    # Get all image files
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    image_files = []
    
    for filename in os.listdir(image_folder):
        if any(filename.lower().endswith(ext) for ext in image_extensions):
            image_files.append(os.path.join(image_folder, filename))
    
    print(f"Found {len(image_files)} images to process...")
    
    if len(image_files) == 0:
        print("No image files found!")
        return
    
    # List to store results
    results = []
    
    # Process each image
    for i, image_path in enumerate(image_files, 1):
        print(f"Processing image {i}/{len(image_files)}: {os.path.basename(image_path)}")
        
        result = calculate_green_view_ratio_and_sky_visibility(image_path)
        if result is not None:
            results.append(result)
        
        # Show progress every 10 images
        if i % 10 == 0:
            print(f"Completed {i}/{len(image_files)} images")
    
    print(f"\nProcessing complete! Successfully processed {len(results)} images")
    
    # Convert results to DataFrame
    if results:
        df_results = pd.DataFrame(results)
        
        # Extract location_id from image name (remove file extension)
        def extract_location_id_from_filename(image_name):
            """Extract location_id from image filename, removing file extension"""
            # Remove file extension (.jpg, .jpeg, .png, etc.)
            location_id = os.path.splitext(image_name)[0]
            return location_id
        
        df_results['location_id'] = df_results['image_name'].apply(extract_location_id_from_filename)
        
        # Reorder columns
        columns_order = [
            'location_id', 'image_name', 'green_view_ratio', 'sky_visibility', 
            'tree_ratio', 'tree_pixels', 'sky_pixels', 'grass_pixels', 'plant_pixels',
            'total_pixels', 'image_width', 'image_height', 'image_path'
        ]
        df_results = df_results[columns_order]
        
        # Display statistics
        print("\n=== Processing Results Statistics ===")
        print(f"Green View Ratio - Mean: {df_results['green_view_ratio'].mean():.2f}%")
        print(f"Green View Ratio - Min: {df_results['green_view_ratio'].min():.2f}%")
        print(f"Green View Ratio - Max: {df_results['green_view_ratio'].max():.2f}%")
        print(f"Sky Visibility - Mean: {df_results['sky_visibility'].mean():.2f}%")
        print(f"Sky Visibility - Min: {df_results['sky_visibility'].min():.2f}%")
        print(f"Sky Visibility - Max: {df_results['sky_visibility'].max():.2f}%")
        
        # Save to CSV file
        output_csv_path = os.path.join(os.path.dirname(image_folder), 'green_view_ratio_and_sky_visibility_results.csv')
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nResults saved to: {output_csv_path}")
        
        # Display first few rows
        print("\n=== First 5 Rows Preview ===")
        print(df_results.head())
        
        return df_results
    else:
        print("No images were processed successfully!")
        return None

print("Batch processing function defined successfully!")

Batch processing function defined successfully!


In [7]:
# Test single image (optional)
# First test one image to ensure the model works properly

def test_single_image():
    """Test processing of a single image"""
    image_folder = r'E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\Selected_Images'
    
    # Get the first image for testing
    image_files = []
    for filename in os.listdir(image_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(image_folder, filename))
    
    if image_files:
        test_image = image_files[0]
        print(f"Test image: {os.path.basename(test_image)}")
        
        # Calculate metrics
        result = calculate_green_view_ratio_and_sky_visibility(test_image)
        if result:
            print("\n=== Test Results ===")
            print(f"Green View Ratio: {result['green_view_ratio']}%")
            print(f"Sky Visibility: {result['sky_visibility']}%")
            print(f"Pure Tree Ratio: {result['tree_ratio']}%")
            
            # Display visualization results
            print("\nGenerating visualization results...")
            visualize_segmentation(test_image, save_visualization=True)
            
            return True
    else:
        print("No test image found!")
        return False

# Run test (uncomment the line below to run the test)
# test_single_image()

In [8]:
# Execute batch processing
# Run this cell to process all images and generate CSV file

print("Starting batch image processing...")
print("=" * 50)

# Run batch processing
results_df = process_all_images_and_generate_csv()

if results_df is not None:
    print("\n" + "=" * 50)
    print("Batch processing complete!")
    print(f"Successfully processed {len(results_df)} images")
    print("CSV file generated with the following fields:")
    print("- location_id: Location ID (filename without extension)")
    print("- image_name: Original image filename")
    print("- green_view_ratio: Green View Ratio (%)")
    print("- sky_visibility: Sky Visibility (%)")
    print("- tree_ratio: Pure Tree Ratio (%)")
    print("- tree_pixels: Number of tree pixels")
    print("- sky_pixels: Number of sky pixels") 
    print("- grass_pixels: Number of grass pixels")
    print("- plant_pixels: Number of plant pixels")
    print("- total_pixels: Total number of pixels")
    print("- image_width: Image width")
    print("- image_height: Image height")
    print("- image_path: Full image path")
else:
    print("Batch processing failed! Please check image folder path and files.")

Starting batch image processing...
Found 900 images to process...
Processing image 1/900: 513e1a00fdc9f035870090bf.jpg
Processing image 2/900: 513e1a01fdc9f035870090c4.jpg
Processing image 3/900: 513e1a04fdc9f035870090cf.jpg
Processing image 4/900: 513e1a05fdc9f035870090d2.jpg
Processing image 5/900: 513e1a05fdc9f035870090d3.jpg
Processing image 6/900: 513e1a07fdc9f035870090d9.jpg
Processing image 7/900: 513e1a09fdc9f035870090dc.jpg
Processing image 8/900: 513e1a0afdc9f035870090de.jpg
Processing image 9/900: 513e1a0bfdc9f035870090e4.jpg
Processing image 10/900: 513e1a0cfdc9f035870090e6.jpg
Completed 10/900 images
Processing image 11/900: 513e1a0dfdc9f035870090ea.jpg
Processing image 12/900: 513e1a0ffdc9f035870090f0.jpg
Processing image 13/900: 513e1a12fdc9f035870090f6.jpg
Processing image 14/900: 513e1a12fdc9f035870090f7.jpg
Processing image 15/900: 513e1a13fdc9f035870090f9.jpg
Processing image 16/900: 513e1a16fdc9f03587009103.jpg
Processing image 17/900: 513e1a16fdc9f03587009105.jpg
P

In [9]:
# View final processing results and statistics
if 'results_df' in globals() and results_df is not None:
    print("Image segmentation processing complete!")
    print("=" * 60)
    print(f"CSV file path: E:\\01_UCL\\08_Dissertation\\02_DataProcessing\\02_ImageSegmentation\\green_view_ratio_and_sky_visibility_results.csv")
    print(f"Successfully processed images: {len(results_df)}")

    print("\nGreen View Ratio Statistics:")
    print(f"  • Mean: {results_df['green_view_ratio'].mean():.2f}%")
    print(f"  • Min: {results_df['green_view_ratio'].min():.2f}%")
    print(f"  • Max: {results_df['green_view_ratio'].max():.2f}%")
    print(f"  • Standard Deviation: {results_df['green_view_ratio'].std():.2f}%")

    print("\nSky Visibility Statistics:")
    print(f"  • Mean: {results_df['sky_visibility'].mean():.2f}%")
    print(f"  • Min: {results_df['sky_visibility'].min():.2f}%")
    print(f"  • Max: {results_df['sky_visibility'].max():.2f}%")
    print(f"  • Standard Deviation: {results_df['sky_visibility'].std():.2f}%")

    print("\nPure Tree Ratio Statistics:")
    print(f"  • Mean: {results_df['tree_ratio'].mean():.2f}%")
    print(f"  • Min: {results_df['tree_ratio'].min():.2f}%")
    print(f"  • Max: {results_df['tree_ratio'].max():.2f}%")
    print(f"  • Standard Deviation: {results_df['tree_ratio'].std():.2f}%")

    print("\nDataset Overview (first 5 rows):")
    print(results_df[['location_id', 'image_name', 'green_view_ratio', 'sky_visibility', 'tree_ratio']].head())

    print("\nCSV file contains the following fields:")
    for i, col in enumerate(results_df.columns, 1):
        print(f"  {i:2d}. {col}")

    print("\nProcessing complete! Now location_id directly uses filename (without .jpg extension)")
else:
    print("Please run the batch processing code first to generate result data!")

Image segmentation processing complete!
CSV file path: E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\green_view_ratio_and_sky_visibility_results.csv
Successfully processed images: 900

Green View Ratio Statistics:
  • Mean: 17.23%
  • Min: 0.00%
  • Max: 90.86%
  • Standard Deviation: 16.97%

Sky Visibility Statistics:
  • Mean: 21.57%
  • Min: 0.00%
  • Max: 50.97%
  • Standard Deviation: 11.16%

Pure Tree Ratio Statistics:
  • Mean: 14.53%
  • Min: 0.00%
  • Max: 69.82%
  • Standard Deviation: 14.56%

Dataset Overview (first 5 rows):
                location_id                    image_name  green_view_ratio  \
0  513e1a00fdc9f035870090bf  513e1a00fdc9f035870090bf.jpg              6.08   
1  513e1a01fdc9f035870090c4  513e1a01fdc9f035870090c4.jpg             14.81   
2  513e1a04fdc9f035870090cf  513e1a04fdc9f035870090cf.jpg              0.23   
3  513e1a05fdc9f035870090d2  513e1a05fdc9f035870090d2.jpg             90.86   
4  513e1a05fdc9f035870090d3  513e1a05fdc9f03

In [10]:
# Update location_id format in existing CSV file
print("Updating location_id format in CSV file...")

# Read existing CSV file
csv_path = r'E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\green_view_ratio_and_sky_visibility_results.csv'
df_updated = pd.read_csv(csv_path)

# Update location_id: use filename without extension
def extract_location_id_from_filename(image_name):
    """Extract location_id from image filename, removing file extension"""
    return os.path.splitext(image_name)[0]

df_updated['location_id'] = df_updated['image_name'].apply(extract_location_id_from_filename)

# Save updated CSV file (overwrite original file)
df_updated.to_csv(csv_path, index=False, encoding='utf-8-sig')

print("CSV file updated successfully!")
print("\nUpdated data example (first 3 rows):")
print(df_updated[['location_id', 'image_name', 'green_view_ratio', 'sky_visibility', 'tree_ratio']].head(3))

# Verify location_id format
print(f"\nVerification results:")
print(f"- location_id example: {df_updated['location_id'].iloc[0]}")
print(f"- image_name example: {df_updated['image_name'].iloc[0]}")
print(f"- Extension correctly removed: {df_updated['location_id'].iloc[0] == os.path.splitext(df_updated['image_name'].iloc[0])[0]}")

# Update global variable for use by other code
results_df = df_updated
print(f"\nUpdate complete! Processed {len(results_df)} rows of data")

Updating location_id format in CSV file...
CSV file updated successfully!

Updated data example (first 3 rows):
                location_id                    image_name  green_view_ratio  \
0  513e1a00fdc9f035870090bf  513e1a00fdc9f035870090bf.jpg              6.08   
1  513e1a01fdc9f035870090c4  513e1a01fdc9f035870090c4.jpg             14.81   
2  513e1a04fdc9f035870090cf  513e1a04fdc9f035870090cf.jpg              0.23   

   sky_visibility  tree_ratio  
0           29.95        5.96  
1           31.63       12.37  
2           32.64        0.12  

Verification results:
- location_id example: 513e1a00fdc9f035870090bf
- image_name example: 513e1a00fdc9f035870090bf.jpg
- Extension correctly removed: True

Update complete! Processed 900 rows of data


# 3. How to Use

## Model Description
I have selected the **SegFormer-B0** model for you, which is trained on the ADE20K dataset and performs excellently in semantic segmentation tasks:

### Reasons for Selection:
1. **Suitable for urban scene analysis**: SegFormer performs excellently in various outdoor scene segmentation
2. **Good compatibility**: Uses safetensors format, avoiding PyTorch version compatibility issues
3. **Strong recognition capability**: Can accurately identify key elements like sky, trees, grass, and plants
4. **High computational efficiency**: B0 version maintains good accuracy while providing efficient performance

### Metric Calculation Methods:
- **Green View Ratio**: Total percentage of tree, grass, and plant pixels
- **Sky Visibility**: Percentage of sky pixels relative to total pixels
- **Tree Ratio**: Percentage of pure tree pixels relative to total pixels

### Usage Steps:
1. **Test single image**: Run `test_single_image()` to view segmentation results
2. **Batch processing**: Run the next cell's code to process all images
3. **View results**: The generated CSV file will contain green view ratio and sky visibility data for all images

## Implementation Details
This notebook implements a complete pipeline for:
- Loading and preprocessing street view images
- Applying semantic segmentation using SegFormer
- Calculating vegetation and sky metrics
- Batch processing multiple images efficiently
- Generating comprehensive CSV output with detailed statistics

In [11]:
# Check image folder and files
image_folder = r'E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\Selected_Images'

print("Checking image folder...")
if os.path.exists(image_folder):
    print(f"Image folder exists: {image_folder}")
    
    # Check image files
    image_files = []
    for filename in os.listdir(image_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            image_files.append(filename)
    
    print(f"Found {len(image_files)} image files")
    if len(image_files) > 0:
        print("First 5 file examples:")
        for i, filename in enumerate(image_files[:5], 1):
            print(f"  {i}. {filename}")
    
    if len(image_files) > 0:
        print("\nYou can now run the following code:")
        print("1. Test single image: test_single_image()")
        print("2. Batch processing: process_all_images_and_generate_csv()")
    else:
        print("No image files found, please check folder path and file formats")
else:
    print(f"Image folder does not exist: {image_folder}")
    print("Please confirm the path is correct")

Checking image folder...
Image folder exists: E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\Selected_Images
Found 900 image files
First 5 file examples:
  1. 513e1a00fdc9f035870090bf.jpg
  2. 513e1a01fdc9f035870090c4.jpg
  3. 513e1a04fdc9f035870090cf.jpg
  4. 513e1a05fdc9f035870090d2.jpg
  5. 513e1a05fdc9f035870090d3.jpg

You can now run the following code:
1. Test single image: test_single_image()
2. Batch processing: process_all_images_and_generate_csv()


In [12]:
# Run single image test (without visualization to avoid kernel crashes)
print("Testing single image...")

def test_single_image_safe():
    """Safely test single image processing (without matplotlib visualization)"""
    image_folder = r'E:\01_UCL\08_Dissertation\02_DataProcessing\02_ImageSegmentation\Selected_Images'
    
    # Get first image for testing
    image_files = []
    for filename in os.listdir(image_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(image_folder, filename))
    
    if image_files:
        test_image = image_files[0]
        print(f"Test image: {os.path.basename(test_image)}")
        
        # Calculate metrics
        result = calculate_green_view_ratio_and_sky_visibility(test_image)
        if result:
            print("\n=== Test Results ===")
            print(f"Green View Ratio: {result['green_view_ratio']}%")
            print(f"Sky Visibility: {result['sky_visibility']}%")
            print(f"Pure Tree Ratio: {result['tree_ratio']}%")
            print(f"Image size: {result['image_width']} x {result['image_height']}")
            print(f"Total pixels: {result['total_pixels']:,}")
            
            print("\n=== Detailed Pixel Statistics ===")
            print(f"Tree pixels: {result['tree_pixels']:,}")
            print(f"Sky pixels: {result['sky_pixels']:,}")
            print(f"Grass pixels: {result['grass_pixels']:,}")
            print(f"Plant pixels: {result['plant_pixels']:,}")
            
            # Skip visualization to avoid kernel crashes
            print("\nNote: Skipped visualization step to avoid kernel crashes.")
            print("If visualization is needed, run the visualization function separately after batch processing.")
            
            return True
        else:
            print("Image processing failed")
            return False
    else:
        print("No test image found!")
        return False

# Run safe test
success = test_single_image_safe()

if success:
    print("\nSingle image test successful! Model is working properly.")
    print("You can now safely run batch processing to process all 900 images.")
else:
    print("Single image test failed, please check model and image paths.")

Testing single image...
Test image: 513e1a00fdc9f035870090bf.jpg

=== Test Results ===
Green View Ratio: 6.08%
Sky Visibility: 29.95%
Pure Tree Ratio: 5.96%
Image size: 400 x 300
Total pixels: 120,000

=== Detailed Pixel Statistics ===
Tree pixels: 7,150
Sky pixels: 35,941
Grass pixels: 0
Plant pixels: 140

Note: Skipped visualization step to avoid kernel crashes.
If visualization is needed, run the visualization function separately after batch processing.

Single image test successful! Model is working properly.
You can now safely run batch processing to process all 900 images.
